# B2 — Clasificador Urbanístico

Notebook de clasificación taxonómica para publicaciones del dominio urbanístico.
Sigue la misma arquitectura incremental que B0 (ambiental-energético) y B1 (hídrico y natural).

In [ ]:
import os
import html
import re
import json
import asyncio
from pathlib import Path

import pandas as pd
import numpy as np

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    hamming_loss,
    jaccard_score,
)
from sklearn.preprocessing import MultiLabelBinarizer

from dotenv import find_dotenv, load_dotenv
from pydantic_ai import Agent

SEED = 29092025

load_dotenv(find_dotenv())

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider

LM_STUDIO_MODEL = "qwen/qwen3.5-9b"

model = OpenAIChatModel(
    LM_STUDIO_MODEL,
    provider=OpenAIProvider(
        base_url="http://localhost:1234/v1",
        api_key="lm-studio",
    ),
)

In [ ]:
import httpx

try:
    r = httpx.get("http://localhost:1234/v1/models", timeout=3)
    modelos = [m["id"] for m in r.json()["data"]]
    assert LM_STUDIO_MODEL in modelos, f"{LM_STUDIO_MODEL} no está cargado en LM Studio"
    print(f"OK: {LM_STUDIO_MODEL} listo  |  otros: {[m for m in modelos if m != LM_STUDIO_MODEL]}")
except httpx.ConnectError:
    raise RuntimeError("LM Studio no responde - arrancar el servidor antes de continuar")

## 1. Schema B2

In [ ]:
from clasificador.schema_B2 import (
    ActType, CategoryTypeB2, SubcategoryTypeB2, ClassifierOutputB2
)

In [ ]:
ejemplo_valido = ClassifierOutputB2(
    is_relevant=True,
    act_type=ActType.RESOLUCION,
    categories=[CategoryTypeB2.LIC_URB],
    subcategories=[SubcategoryTypeB2.USO_RUSTICO, SubcategoryTypeB2.USO_ENERGETICO],
    confidence=0.95,
    reasoning="'autorización de uso excepcional de suelo rústico' → LIC_URB + uso_rustico. 'planta solar fotovoltaica' → uso_energetico.",
)
print("Ejemplo válido:")
print(ejemplo_valido.model_dump_json(indent=2))

print("\nViolación de invariante:")
try:
    ClassifierOutputB2(
        is_relevant=False, act_type=ActType.RESOLUCION,
        categories=[CategoryTypeB2.PGOU], subcategories=[],
        confidence=0.5, reasoning="Prueba.",
    )
except Exception as e:
    print(f"  ValidationError -> {e.errors()[0]['msg']}")

## 2. Exploración del corpus B2

In [ ]:
from clasificador.agent import get_ambito, inferir_act_type

PATH_PARQUET = "../data/raw/silver_official_gazettes_2025_Q1.parquet"
df = pd.read_parquet(PATH_PARQUET)
df["description"] = df["description"].apply(lambda x: html.unescape(str(x)) if pd.notna(x) else x)
print(f"Corpus total: {len(df):,} registros | Columnas: {list(df.columns)}")

In [ ]:
df["act_type_n1"] = df.apply(
    lambda r: inferir_act_type(str(r["description"]), str(r["bulletin"])).value, axis=1
)

keywords_dominio_B2 = [
    "plan general de ordenación", "pgou", "poum", "pom de ", "pxom", "normas subsidiarias",
    "normas urbanísticas municipales", "plan parcial", "plan especial", "estudio de detalle",
    "modificación puntual", "proyecto de urbanización", "reparcelación",
    "licencia urbanística", "autorización de uso excepcional", "calificación urbanística",
    "actuación específica de interés público", "declaración de interés comunitario",
    "declaración de utilidad e interés social",
]
mask_b2 = df["description"].str.lower().str.contains("|".join(keywords_dominio_B2), na=False)
df_b2 = df[mask_b2].copy()
print(f"Universo B2 estimado: {len(df_b2):,} registros ({len(df_b2)/len(df)*100:.2f}% del corpus)")
print(f"\nDistribución por boletín (top 10):")
print(df_b2["bulletin"].value_counts().head(10).to_string())
print(f"\nDistribución N1 en universo B2:")
print(df_b2["act_type_n1"].value_counts().head(8).to_string())

## 3. Ground truth — Muestreo estratificado

In [ ]:
keywords_B2 = {
    "PGOU":     ["plan general de ordenación", "pgou", "poum", "pom de ", "pxom",
                 "pgom", "pgm de ", "normas subsidiarias",
                 "normas urbanísticas municipales", " num "],
    "PLAN_ESP": ["plan parcial", "plan especial", "estudio de detalle"],
    "MOD_PUN":  ["modificación puntual", "modificación del plan general",
                 "modificación de las normas subsidiarias",
                 "modificación de las normas urbanísticas"],
    "PROY_URB": ["proyecto de urbanización", "reparcelación", "parcelación",
                 "unidad de ejecución"],
    "LIC_URB":  ["licencia urbanística", "autorización de uso excepcional",
                 "calificación urbanística",
                 "actuación específica de interés público",
                 "usos y actividades admisibles en suelo rústico",
                 "autorización de actividades en suelo no urbanizable"],
    "DIC_INT":  ["declaración de interés comunitario",
                 "declaración de utilidad e interés social",
                 "uso de interés general"],
}

cuotas_B2 = {
    "LIC_URB": 30, "MOD_PUN": 25, "PLAN_ESP": 25,
    "PGOU": 25, "PROY_URB": 15, "DIC_INT": 5,
}
N_MULTILABEL_FV = 10
N_NEGATIVOS     = 15

desc_lower = df["description"].str.lower()
print(f"{'Label':<12} {'Pool':>8} {'Cuota':>7} {'Estado':>12}")
print("-" * 44)
for label, kws in keywords_B2.items():
    mask = desc_lower.str.contains("|".join(kws), na=False)
    pool = mask.sum()
    cuota = cuotas_B2.get(label, 0)
    estado = "OK" if pool >= cuota else f"REDUCIDA a {min(cuota, pool)}"
    print(f"{label:<12} {pool:>8,} {cuota:>7} {estado:>12}")

In [ ]:
desc_lower = df["description"].str.lower()
sampled_ids = set()
frames = []

# Keywords para detectar renovables en suelo rústico (multilabel B2+B0)
kws_fv_rustico = ["fotovoltaica", "eólica", "aerogenerador", "solar"]
kws_rustico    = ["suelo rústico", "suelo no urbanizable", "uso excepcional"]

# 1. Muestra estratificada por categoría
for label, kws in keywords_B2.items():
    mask = desc_lower.str.contains("|".join(kws), na=False)
    mask = mask & ~df.index.isin(sampled_ids)
    pool = df[mask]
    n = min(cuotas_B2.get(label, 0), len(pool))
    if n == 0:
        print(f"  {label:<12} SKIP (pool vacío)")
        continue
    sample = pool.sample(n, random_state=SEED).copy()
    sample["grupo_muestreo"] = label
    sampled_ids.update(sample.index.tolist())
    frames.append(sample)
    print(f"  {label:<12} pool={len(pool):>5,}  sampled={n}")

# 2. Multilabel FV en suelo rústico (LIC_URB solar/eólica)
mask_fv = (
    desc_lower.str.contains("|".join(kws_fv_rustico), na=False)
    & desc_lower.str.contains("|".join(kws_rustico), na=False)
    & ~df.index.isin(sampled_ids)
)
pool_fv = df[mask_fv]
n_fv = min(N_MULTILABEL_FV, len(pool_fv))
if n_fv > 0:
    sample_fv = pool_fv.sample(n_fv, random_state=SEED).copy()
    sample_fv["grupo_muestreo"] = "MULTILABEL_FV"
    sampled_ids.update(sample_fv.index.tolist())
    frames.append(sample_fv)
    print(f"  {'MULTILABEL_FV':<12} pool={len(pool_fv):>5,}  sampled={n_fv}")

# 3. Negativos
all_kws_b2 = [kw for kws in keywords_B2.values() for kw in kws]
mask_neg = (
    ~desc_lower.str.contains("|".join(all_kws_b2), na=False)
    & ~df.index.isin(sampled_ids)
)
pool_neg = df[mask_neg]
negativos = pool_neg.sample(N_NEGATIVOS, random_state=SEED).copy()
negativos["grupo_muestreo"] = "NEGATIVO"
frames.append(negativos)

df_muestreo = pd.concat(frames, ignore_index=True)
df_muestreo["id"] = range(len(df_muestreo))
print(f"\nTotal muestreado: {len(df_muestreo)} registros")
print(df_muestreo["grupo_muestreo"].value_counts().to_string())

In [ ]:
print("Validación del muestreo - 3 ejemplos por grupo:\n")
for grupo in df_muestreo["grupo_muestreo"].unique():
    muestra = df_muestreo[df_muestreo["grupo_muestreo"] == grupo].head(3)
    print(f"--- {grupo} ---")
    for _, row in muestra.iterrows():
        desc = str(row["description"])[:130].replace("\n", " ")
        bul  = str(row.get("bulletin", "?")).upper()
        print(f"  [{bul}] {desc}")
    print()

In [ ]:
PATH_MUESTREO = "../data/ground_truth/ground_truth_B2_muestreo.csv"
Path(PATH_MUESTREO).parent.mkdir(parents=True, exist_ok=True)

df_muestreo["is_relevant_gt"]   = ""
df_muestreo["categories_gt"]    = ""
df_muestreo["subcategories_gt"] = ""
df_muestreo["notas_anotador"]   = ""

df_muestreo[["id","bulletin","description","grupo_muestreo",
             "is_relevant_gt","categories_gt","subcategories_gt","notas_anotador"]].to_csv(
    PATH_MUESTREO, index=False
)
print(f"Guardado: {PATH_MUESTREO}  ({len(df_muestreo)} registros)")
print("Siguiente paso: anotar manualmente las columnas is_relevant_gt, categories_gt, subcategories_gt")

## 4. Agente base

In [ ]:
from clasificador.schema_B2 import ClassifierOutputB2
from clasificador.prompts_B2 import PROMPT_REGISTRY_B2
from clasificador.agent import build_agent, run_experiment

In [ ]:
agent_b2_v1 = build_agent(
    model, "v1",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

# Casos cualitativos representativos del dominio B2
casos_b2 = [
    ("Anuncio del Ayuntamiento de Torrelodones por el que se somete a información pública la aprobación inicial del Plan General de Ordenación Urbana.", "bocm"),
    ("Resolución de la Consejería de Urbanismo por la que se aprueba definitivamente la modificación puntual número 5 del Plan General Municipal de Cuenca.", "docm"),
    ("Anuncio de información pública relativo a la solicitud de autorización de uso excepcional de suelo rústico para instalación solar fotovoltaica de 2 MW en Villarrobledo (Albacete).", "docm"),
    ("Anuncio del Ajuntament de Girona pel qual se sotmet a informació pública l'aprovació inicial del Pla d'Ordenació Urbanística Municipal (POUM) de Girona.", "dogc"),
    ("Resolución de la Universidad Complutense de Madrid por la que se aprueba la modificación del plan de estudios del Grado en Medicina conforme al Real Decreto 822/2021.", "boe"),
]

for desc, bul in casos_b2:
    print(f"\n[{bul.upper()}] {desc[:80]}...")
    # result = await agent_b2_v1.run(f"Boletín: {bul.upper()}\n\nDescripción: {desc}")
    # print(result.output.model_dump_json(indent=2))

## 5. Funciones de evaluación B2

In [ ]:
def parse_labels(value) -> set:
    """Convierte cualquier representación de etiquetas a un set de strings."""
    if pd.isna(value) or str(value).strip() in ("", "nan"):
        return set()
    s = str(value).strip()
    if s.startswith("["):
        try:
            items = json.loads(s)
            return {str(i).strip('"') for i in items if i}
        except json.JSONDecodeError:
            pass
    return {v.strip().strip('"') for v in s.split(",") if v.strip()}

In [ ]:
def compute_metrics_B2(df_eval, label="", verbose=True):
    """
    Calcula métricas multilabel para el clasificador B2.
    Columnas esperadas: is_relevant_gt, categories_gt, is_relevant_pred, categories_pred.
    """
    ALL_CATS_B2 = [e.value for e in CategoryTypeB2]
    mlb = MultiLabelBinarizer(classes=ALL_CATS_B2)
    mlb.fit([ALL_CATS_B2])

    gt_labels   = [parse_labels(v) & set(ALL_CATS_B2) for v in df_eval["categories_gt"]]
    pred_labels = [parse_labels(v) & set(ALL_CATS_B2) for v in df_eval["categories_pred"]]
    Y    = mlb.transform(gt_labels)
    Yhat = mlb.transform(pred_labels)

    is_rel_gt   = df_eval["is_relevant_gt"].astype(bool)
    is_rel_pred = df_eval["is_relevant_pred"].astype(bool)

    exact = pd.Series([set(g) == set(p) for g, p in zip(gt_labels, pred_labels)])
    rel   = is_rel_gt

    metrics = {
        "is_rel_accuracy":  round(accuracy_score(is_rel_gt, is_rel_pred), 4),
        "is_rel_precision": round(precision_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_recall":    round(recall_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "is_rel_f1":        round(f1_score(is_rel_gt, is_rel_pred, zero_division=0), 4),
        "micro_f1":         round(f1_score(Y, Yhat, average="micro", zero_division=0), 4),
        "macro_f1":         round(f1_score(Y, Yhat, average="macro", zero_division=0), 4),
        "hamming_loss":     round(hamming_loss(Y, Yhat), 4),
        "jaccard_samples":  round(jaccard_score(Y, Yhat, average="samples", zero_division=0), 4),
        "subset_accuracy":  round(accuracy_score(Y, Yhat), 4),
        "exact": exact,
        "rel":   rel,
    }

    if verbose:
        title = f"-- {label} --" if label else "-- Métricas B2 --"
        print(f"\n{title}\n")
        tp = int((is_rel_gt & is_rel_pred).sum())
        fp = int((~is_rel_gt & is_rel_pred).sum())
        fn = int((is_rel_gt & ~is_rel_pred).sum())
        tn = int((~is_rel_gt & ~is_rel_pred).sum())
        print(f"is_relevant  Acc={metrics['is_rel_accuracy']:.3f}  P={metrics['is_rel_precision']:.3f}  "
              f"R={metrics['is_rel_recall']:.3f}  F1={metrics['is_rel_f1']:.3f}")
        print(f"             TP={tp}  FP={fp}  FN={fn}  TN={tn}\n")
        print(f"N2 multilabel:")
        print(f"  Micro F1:      {metrics['micro_f1']:.3f}")
        print(f"  Macro F1:      {metrics['macro_f1']:.3f}")
        print(f"  Hamming Loss:  {metrics['hamming_loss']:.4f}")
        print(f"  Jaccard:       {metrics['jaccard_samples']:.3f}")
        print(f"  Subset Acc:    {metrics['subset_accuracy']:.3f}\n")
        f1s = f1_score(Y, Yhat, average=None, zero_division=0)
        sups = Y.sum(axis=0)
        print(f"  {'Label':<12} {'P':>6} {'R':>6} {'F1':>6} {'Sup':>5}")
        print(f"  {'-'*37}")
        for i, cat in enumerate(ALL_CATS_B2):
            prec = precision_score(Y[:, i], Yhat[:, i], zero_division=0)
            rec  = recall_score(Y[:, i], Yhat[:, i], zero_division=0)
            print(f"  {cat:<12} {prec:>6.3f} {rec:>6.3f} {f1s[i]:>6.3f} {int(sups[i]):>5}")
        print(f"  {'-'*37}")
        print(f"  {'Macro':<12} {'':>6} {'':>6} {metrics['macro_f1']:>6.3f}\n")

        cards = [len(g) for g in gt_labels]
        print(f"  Subset Acc por cardinalidad:")
        for card in [0, 1, 2]:
            idx = [i for i, c in enumerate(cards) if c == card]
            if idx:
                acc = accuracy_score(Y[idx], Yhat[idx])
                print(f"    card={card} ({'no relevante' if card == 0 else str(card)}) : {acc:.3f}  ({int(acc*len(idx))}/{len(idx)})")
        idx3 = [i for i, c in enumerate(cards) if c >= 3]
        if idx3:
            acc3 = accuracy_score(Y[idx3], Yhat[idx3])
            print(f"    card>=3               : {acc3:.3f}  ({int(acc3*len(idx3))}/{len(idx3)})")

        conf = df_eval.get("confidence", pd.Series(dtype=float))
        if conf.notna().any():
            print(f"\n  Confianza: media={conf.mean():.3f}  min={conf.min():.3f}  max={conf.max():.3f}")

    return metrics

In [ ]:
def print_errors_B2(df_eval, exact, rel, label="", n=None):
    errores = df_eval[~exact & rel]
    if n is not None:
        errores = errores.head(n)
    lbl = f" · {label}" if label else ""
    print(f"\n-- Errores N2 en relevantes{lbl} --")
    print(f"Total: {len(errores)}\n")
    for _, row in errores.iterrows():
        gt   = sorted(parse_labels(row["categories_gt"]))
        pred = sorted(parse_labels(row["categories_pred"]))
        falta = sorted(set(gt) - set(pred))
        sobra = sorted(set(pred) - set(gt))
        desc  = str(row["description"])[:90].replace("\n", " ")
        razon = str(row.get("reasoning", "")).replace("\n", " ")[:120]
        print(f"ID {row['id']} | GT={gt} | PRED={pred}")
        print(f"  Falta: {falta} | Sobra: {sobra}")
        print(f"  {desc}...")
        print(f"  Razonamiento: {razon}")
        print()

---

##  6. Experimento 1 - Baseline zero-shot

**Problema**: No existe un clasificador para el dominio urbanístico. Necesitamos una línea base que mida el rendimiento zero-shot antes de cualquier optimización.

**Objetivo**: Establecer el Macro-F1 de referencia con el prompt mínimo operativo (V1) y detectar los patrones de error sistematicos que guiarán la mejora del prompt.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V1 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
df_anotado = pd.read_csv("../data/ground_truth/ground_truth_B2_anotado.csv")
df_anotado_run = df_anotado[df_anotado["is_relevant_gt"].notna()].copy()

df_b2_exp1 = await run_experiment(
    agent_b2_v1, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp1_baseline_qwen9b.csv",
    desc="B2 Exp1 - Baseline V1",
)
df_b2_exp1.head(3)

In [ ]:
df_b2_exp1 = pd.read_csv("../results/b2_exp1_baseline_qwen9b.csv")
df_eval_b2_1 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp1[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_1 = compute_metrics_B2(df_eval_b2_1, "Experimento B2-1 - Baseline")
print_errors_B2(df_eval_b2_1, m_b2_1["exact"], m_b2_1["rel"], label="B2 Experimento 1 - Baseline")

---

##  7. Experimento 2 - Prompt v2 (variantes regionales)

**Problema**: (rellenar tras el análisis de errores del §6)

**Objetivo**: Verificar si añadir la tabla de equivalencias regionales (POUM=PGOU, POM=PGOU, etc.) y la regla EMOT navarro mejora el F1 de PGOU.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V2 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
agent_b2_v2 = build_agent(
    model, "v2",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp2 = await run_experiment(
    agent_b2_v2, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp2_promptv2_qwen9b.csv",
    desc="B2 Exp2 - Prompt v2",
)
df_b2_exp2.head(3)

In [ ]:
df_b2_exp2 = pd.read_csv("../results/b2_exp2_promptv2_qwen9b.csv")
df_eval_b2_2 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp2[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_2 = compute_metrics_B2(df_eval_b2_2, "Experimento B2-2 - Prompt v2")
print_errors_B2(df_eval_b2_2, m_b2_2["exact"], m_b2_2["rel"], label="B2 Experimento 2 - Prompt v2")

---

##  8. Experimento 3 - Prompt v3 (reglas de frontera)

**Problema**: (rellenar tras el análisis de errores del §7)

**Objetivo**: Verificar si las reglas de frontera (MOD_PUN≠PGOU, DIC_INT≠DUP, LIC_URB con renovables, PLAN_ESP≠PGOU) reducen los errores de clasificacion.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V3 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
agent_b2_v3 = build_agent(
    model, "v3",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp3 = await run_experiment(
    agent_b2_v3, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp3_promptv3_qwen9b.csv",
    desc="B2 Exp3 - Prompt v3",
)
df_b2_exp3.head(3)

In [ ]:
df_b2_exp3 = pd.read_csv("../results/b2_exp3_promptv3_qwen9b.csv")
df_eval_b2_3 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp3[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_3 = compute_metrics_B2(df_eval_b2_3, "Experimento B2-3 - Prompt v3")
print_errors_B2(df_eval_b2_3, m_b2_3["exact"], m_b2_3["rel"], label="B2 Experimento 3 - Prompt v3")

---

##  9. Experimento 4 - Prompt v4 (few-shot)

**Problema**: (rellenar tras el análisis de errores del §8)

**Objetivo**: Verificar si 5 ejemplos few-shot sobre los casos mas difíciles (MOD_PUN, PGOU variante regional, LIC_URB solar, falso positivo universidad, EMOT) eliminan los errores residuales.

**Enfoque**: Qwen 3.5 9B · SYSTEM_PROMPT_B2_V4 (V3 + 5 ejemplos) · few-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
agent_b2_v4 = build_agent(
    model, "v4",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_exp4 = await run_experiment(
    agent_b2_v4, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp4_promptv4_qwen9b.csv",
    desc="B2 Exp4 - Prompt v4 few-shot",
)
df_b2_exp4.head(3)

In [ ]:
df_b2_exp4 = pd.read_csv("../results/b2_exp4_promptv4_qwen9b.csv")
df_eval_b2_4 = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_exp4[["description","is_relevant_pred","act_type_pred","categories_pred",
                "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_4 = compute_metrics_B2(df_eval_b2_4, "Experimento B2-4 - Prompt v4 few-shot")
print_errors_B2(df_eval_b2_4, m_b2_4["exact"], m_b2_4["rel"], label="B2 Experimento 4 - Prompt v4 few-shot")

---

##  10. Comparativa de modelos - Gemma 4B con prompt V3

**Problema**: Todos los experimentos anteriores usan Qwen 3.5 9B. No sabemos si el prompt es transferible a modelos mas pequeños y rapidos.

**Objetivo**: Comprobar si Gemma 4 4B con V3 (no V4 por ventana de contexto) alcanza un rendimiento comparable al 9B.

**Enfoque**: Gemma 4 4B · SYSTEM_PROMPT_B2_V3 · zero-shot · sin contexto N1.

**Resultados**: pendiente.

In [ ]:
from pydantic_ai.models.openai import OpenAIChatModel
from pydantic_ai.providers.openai import OpenAIProvider as OAIProvider

# Cargar gemma-4-e4b-it en LM Studio antes de ejecutar
model_gemma = OpenAIChatModel(
    "gemma-4-e4b-it",
    provider=OAIProvider(base_url="http://localhost:1234/v1", api_key="lm-studio")
)
agent_b2_gemma = build_agent(
    model_gemma, "v3",
    output_type=ClassifierOutputB2,
    prompt_registry=PROMPT_REGISTRY_B2,
)

df_b2_gemma = await run_experiment(
    agent_b2_gemma, df_anotado_run,
    use_n1_context=False, concurrency=1,
    output_path="../results/b2_exp_gemma4b.csv",
    desc="B2 Gemma4B - Prompt v3",
)
df_b2_gemma.head(3)

In [ ]:
df_b2_gemma = pd.read_csv("../results/b2_exp_gemma4b.csv")
df_eval_b2_gemma = df_anotado[["id","is_relevant_gt","categories_gt","subcategories_gt","description"]].merge(
    df_b2_gemma[["description","is_relevant_pred","act_type_pred","categories_pred",
                 "subcategories_pred","confidence","reasoning"]], on="description", how="left"
)
m_b2_gemma = compute_metrics_B2(df_eval_b2_gemma, "B2 Gemma 4B - Prompt v3")
print_errors_B2(df_eval_b2_gemma, m_b2_gemma["exact"], m_b2_gemma["rel"], label="B2 Gemma 4B - Prompt v3")

## 11. Tabla resumen - Comparativa de experimentos B2

In [ ]:
experimentos_cfg_B2 = [
    ("B2 Exp1 - Baseline V1",    "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp1_baseline_qwen9b.csv"),
    ("B2 Exp2 - Prompt V2",      "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp2_promptv2_qwen9b.csv"),
    ("B2 Exp3 - Prompt V3",      "Qwen 3.5 9B", "Zero-shot", "../results/b2_exp3_promptv3_qwen9b.csv"),
    ("B2 Exp4 - Prompt V4",      "Qwen 3.5 9B", "Few-shot",  "../results/b2_exp4_promptv4_qwen9b.csv"),
    ("B2 Exp5 - Gemma 4B V3",    "Gemma 4 4B",  "Zero-shot", "../results/b2_exp_gemma4b.csv"),
]

rows_b2 = []
metrics_list_b2 = []
for nombre, modelo, config, path in experimentos_cfg_B2:
    df_r = pd.read_csv(path)
    df_e = df_anotado[["id","is_relevant_gt","categories_gt","description"]].merge(
        df_r[["description","is_relevant_pred","categories_pred","confidence","duration_s"]],
        on="description", how="left"
    )
    m = compute_metrics_B2(df_e, verbose=False)
    metrics_list_b2.append({"Experimento": nombre, **m})
    rows_b2.append({
        "Experimento":     nombre,
        "Modelo":          modelo,
        "Config":          config,
        "is_rel_f1":       m["is_rel_f1"],
        "micro_f1":        m["micro_f1"],
        "macro_f1":        m["macro_f1"],
        "hamming_loss":    m["hamming_loss"],
        "jaccard_samples": m["jaccard_samples"],
        "subset_accuracy": m["subset_accuracy"],
        "mean_duration_s": round(df_r["duration_s"].mean(), 2) if "duration_s" in df_r.columns else None,
        "total_duration_s": round(df_r["duration_s"].sum(), 0) if "duration_s" in df_r.columns else None,
    })

df_summary_b2 = pd.DataFrame(rows_b2)
df_display_b2 = df_summary_b2.copy()
df_display_b2.columns = [
    "Experimento", "Modelo", "Config",
    "is_rel F1", "Micro F1", "Macro F1", "Hamming", "Jaccard", "Subset Acc",
    "s/item", "Total (s)",
]
display(df_display_b2.set_index("Experimento"))